# Installation guide for HLS4ML

Code, files and everything needed is avaible on GitHub under https://github.com/fprotopapa/hls4ml-tutorial.git

### Activate virtual enviroment

Now to start a jupyter session type 
```
jupyter-lab
```
and select the enviroment in the upper right corner.

Otherwise just activate the virtual enviroment with
```
source venv_hls4ml/bin/activate
```
And to check if everything worked:

In [1]:
!which python

/home/fabbio/ProjectsLocal/tutorial-hls4ml/hls4ml/venv_hls4ml/bin/python


### Installation

Now we are able to work inside the new enviroment. 
The first thing to do, is upgrading the packetmanager though.

In [2]:
!pip install --upgrade pip

And finally we can install HLS4ML. 
With the profiling flag we'll install few more dependencies, which are needed later on.

In [3]:
!pip install hls4ml[profiling]

  Using cached hls4ml-0.4.0-py3-none-any.whl (215 kB)
  Using cached onnx-1.8.0-cp36-cp36m-manylinux2010_x86_64.whl (7.7 MB)
  Using cached h5py-3.1.0-cp36-cp36m-manylinux1_x86_64.whl (4.0 MB)
     |████████████████████████████████| 14.8 MB 5.2 MB/s eta 0:00:01
  Using cached cached_property-1.5.2-py2.py3-none-any.whl (7.6 kB)
  Using cached matplotlib-3.3.3-cp36-cp36m-manylinux1_x86_64.whl (11.6 MB)
  Using cached cycler-0.10.0-py2.py3-none-any.whl (6.5 kB)
  Using cached kiwisolver-1.3.1-cp36-cp36m-manylinux1_x86_64.whl (1.1 MB)
     |████████████████████████████████| 2.2 MB 5.9 MB/s eta 0:00:01
  Using cached pandas-1.1.5-cp36-cp36m-manylinux1_x86_64.whl (9.5 MB)
  Using cached protobuf-3.14.0-cp36-cp36m-manylinux1_x86_64.whl (1.0 MB)
  Using cached PyYAML-5.3.1.tar.gz (269 kB)
  Using cached seaborn-0.11.1-py3-none-any.whl (285 kB)
  Using cached scipy-1.5.4-cp36-cp36m-manylinux1_x86_64.whl (25.9 MB)
  Using cached tensorflow-2.4.0-cp36-cp36m-manylinux2010_x86_64.whl (394.7 MB)
  U

## A first small example

First things first. 

In [5]:
%mkdir tutorial-hls4ml/example 
%cd tutorial-hls4ml/example

/home/fabbio/ProjectsLocal/tutorial-hls4ml/hls4ml/tutorial-hls4ml/example


Let's import the hls4ml library.

In [6]:
import hls4ml

And fetch a ready-to-use keras model.

To get a list of avaible models we can use the command below. 
All these models can be used out-of-the-box just by changing the name in the fetch_example_model()-methode.

In [8]:
hls4ml.utils.fetch_example_list()

{   'keras': [   'KERAS_3layer_binarydense_relu_max.json',
                 'keras_bnn.json',
                 'jetTagger_Conv2D_Small.json',
                 'KERAS_conv2d_model.json',
                 'KERAS_dense_16x100x100x100x100x100x5.json',
                 'qkeras_mnist_dense.json',
                 'KERAS_dense_16x200x200x200x200x200x5.json',
                 'KERAS_3layer_ternary_small.json',
                 'qkeras_3layer.json',
                 'KERAS_conv1d_small.json',
                 'jetTagger_Conv2D_Small_NoBatchNorm.json',
                 'KERAS_3layer_binary_smaller.json',
                 'KERAS_conv1d.json',
                 'garnet_1layer.json',
                 'KERAS_3layer.json',
                 'KERAS_1layer.json',
                 'KERAS_3layer_batch_norm.json',
                 'garnet_3layer.json',
                 'KERAS_dense_16x500x500x500x500x500x5.json'],
    'onnx': [   'three_layer_bn_keras.onnx',
                'two_layer_pytorch.onnx',
       

In [7]:
config = hls4ml.utils.fetch_example_model('KERAS_3layer.json')
print(config)

{'OutputDir': 'my-hls-test', 'ProjectName': 'myproject', 'XilinxPart': 'xcku115-flvb2104-2-i', 'ClockPeriod': 5, 'Backend': 'Vivado', 'IOType': 'io_parallel', 'HLSConfig': {'Model': {'Precision': 'ap_fixed<16,6>', 'ReuseFactor': '1'}}, 'KerasJson': 'KERAS_3layer.json', 'KerasH5': 'KERAS_3layer_weights.h5'}


The output shows some predefined settings for creating a hls version of this model. 
So we can adjust the bitwidth, reuse factor, the io type ... you name it. 
This is something we'll look at later. 

To convert the configuration file and the models specified in it to a hls representation, we need to run:

In [9]:
hls_model = hls4ml.converters.keras_to_hls(config)

Interpreting Model
Topology:
Layer name: input_1, layer type: InputLayer, current shape: [[None, 16]]
Layer name: fc1_relu, layer type: Dense, current shape: [[None, 16]]
Layer name: fc2_relu, layer type: Dense, current shape: [[None, 64]]
Layer name: fc3_relu, layer type: Dense, current shape: [[None, 32]]
Layer name: output_softmax, layer type: Dense, current shape: [[None, 32]]
Creating HLS model


To use the build-methode we have to make sure that the vivado path is avaible.

In [10]:
import os

os.environ['PATH'] = '/tools/Xilinx/Vivado/2019.2/bin:' + os.environ['PATH']

And now let's build the project!
This will take some time...

To monitor the logs run the command below inside a new console:
```
tail -f my-hls-test/vivado_hls.log
```

In [11]:
hls_model.build()

Writing HLS project
Done


To check the resources needed and latency, have a look at the report.

In [13]:
hls4ml.report.read_vivado_report('my-hls-test')

Found 1 solution(s) in my-hls-test/myproject_prj.
Reports for solution "solution1":

C SIMULATION RESULT:
INFO: [SIM 2] *************** CSIM start ***************
INFO: [SIM 4] CSIM will launch GCC as the compiler.
   Compiling ../../../../myproject_test.cpp in debug mode
   Compiling ../../../../firmware/myproject.cpp in debug mode
   Generating csim.exe
INFO: Unable to open input/predictions file, using default input.
0.0292969 0.756836 0.0546875 0.139648 0.0371094 
INFO: Saved inference results to file: tb_data/csim_results.log
INFO: [SIM 1] CSim done with 0 errors.
INFO: [SIM 3] *************** CSIM finish ***************

SYNTHESIS REPORT:
== Vivado HLS Report for 'myproject'
* Date:           Thu Jan  7 15:06:35 2021

* Version:        2019.2 (Build 2704478 on Wed Nov 06 22:10:23 MST 2019)
* Project:        myproject_prj
* Solution:       solution1
* Product family: kintexu
* Target device:  xcku115-flvb2104-2-i


== Performance Estimates
+ Timing: 
    * Summary: 
    +--------+

Great, it worked!

Now let's build a model from scratch.

Continue with 2_Create_a_keras_model.